# 01 - Data Profiling Awal

**Nama:** Alfito Afdhan Nugraha  
**NIM:** 202310370311415  
**Dataset:** Prakiraan Cuaca Terbuka BMKG (lihat `data/README.md`).  
**Tanggal:** 2026-10-02

---
Notebook ini melakukan *data profiling* awal terhadap data prakiraan cuaca BMKG yang sudah diunduh ke `data/raw/*.parquet`.
Jalankan sel secara berurutan dari atas ke bawah.

In [ ]:
import glob
import polars as pl
import plotly.express as px

# Cek versi library
print('Polars:', pl.__version__)

files = sorted(glob.glob('../data/raw/*.parquet'))
print(f'\nJumlah file Parquet: {len(files)}')
for f in files:
    print(' -', f)

## 1. Ukuran Data

In [ ]:
# Scan lazy (hemat memori)
lf = pl.scan_parquet('../data/raw/*.parquet')

n_rows = lf.select(pl.len()).collect().item()
n_cols = len(lf.collect_schema())
print(f'Jumlah baris  : {n_rows:,}')
print(f'Jumlah kolom  : {n_cols}')

## 2. Skema & Tipe Data

In [ ]:
schema = lf.collect_schema()
schema_df = pl.DataFrame({
    'kolom': list(schema.names()),
    'tipe': [str(t) for t in schema.dtypes()]
})
print(schema_df)

## 3. Contoh Isi Data (5 Baris Pertama)

In [ ]:
df_sample = lf.head(5).collect()
print(df_sample)

## 4. Nilai Kosong (Null) per Kolom

In [ ]:
null_df = (
    lf.null_count()
    .collect()
    .transpose(include_header=True, column_names=['null_count'])
    .with_columns(
        (pl.col('null_count') / n_rows * 100).round(2).alias('null_pct')
    )
    .sort('null_count', descending=True)
)
print(null_df)

## 5. Statistik Deskriptif (Kolom Numerik)

In [ ]:
# Collect sebagian untuk describe (tidak butuh semua data)
df = lf.collect()
print(df.describe())

## 6. Cakupan Wilayah & Waktu

In [ ]:
coverage = df.select(
    pl.col('provinsi').n_unique().alias('n_provinsi'),
    pl.col('kotkab').n_unique().alias('n_kotkab'),
    pl.col('kecamatan').n_unique().alias('n_kecamatan'),
    pl.col('kode_adm4').n_unique().alias('n_desa'),
)
print('Cakupan wilayah:')
print(coverage)

# Daftar provinsi
if 'provinsi' in df.columns:
    print('\nProvinsi yang ada dalam data:')
    print(df['provinsi'].unique().sort().to_list())

## 7. Distribusi Suhu (t)

In [ ]:
if 't' in df.columns:
    df_t = df.select('t').drop_nulls()
    fig = px.histogram(
        df_t.to_pandas(),
        x='t',
        nbins=40,
        title='Distribusi Suhu Prakiraan (°C) - Data BMKG',
        labels={'t': 'Suhu (°C)'},
        color_discrete_sequence=['#0077B6']
    )
    fig.update_layout(bargap=0.05)
    fig.show()
    fig.write_image('../output/figures/distribusi_suhu.png')
    print('Gambar disimpan ke output/figures/distribusi_suhu.png')
else:
    print('Kolom "t" tidak ditemukan. Periksa nama kolom pada schema di atas.')

## 8. Distribusi Kelembapan (hu)

In [ ]:
if 'hu' in df.columns:
    df_hu = df.select('hu').drop_nulls()
    fig = px.histogram(
        df_hu.to_pandas(),
        x='hu',
        nbins=40,
        title='Distribusi Kelembapan Relatif (%) - Data BMKG',
        labels={'hu': 'Kelembapan (%)'},
        color_discrete_sequence=['#48CAE4']
    )
    fig.update_layout(bargap=0.05)
    fig.show()
    fig.write_image('../output/figures/distribusi_kelembapan.png')
    print('Gambar disimpan ke output/figures/distribusi_kelembapan.png')
else:
    print('Kolom "hu" tidak ditemukan.')

## 9. Distribusi Deskripsi Cuaca

In [ ]:
desc_col = None
for col in ['weather_desc', 'weather_desc_en', 'cuaca']:
    if col in df.columns:
        desc_col = col
        break

if desc_col:
    weather_count = (
        df.group_by(desc_col)
        .agg(pl.len().alias('count'))
        .sort('count', descending=True)
        .head(15)
    )
    fig = px.bar(
        weather_count.to_pandas(),
        x='count', y=desc_col,
        orientation='h',
        title='Top 15 Deskripsi Cuaca (Prakiraan BMKG)',
        labels={desc_col: 'Kondisi Cuaca', 'count': 'Frekuensi'},
        color='count',
        color_continuous_scale='Blues'
    )
    fig.update_layout(yaxis={'categoryorder': 'total ascending'})
    fig.show()
    fig.write_image('../output/figures/distribusi_cuaca.png')
    print('Gambar disimpan ke output/figures/distribusi_cuaca.png')
else:
    print('Kolom deskripsi cuaca tidak ditemukan. Cek nama kolom pada schema.')

## 10. Deteksi Outlier Sederhana (IQR)

In [ ]:
numeric_cols = [col for col, dtype in df.schema.items()
                if dtype in (pl.Float32, pl.Float64, pl.Int32, pl.Int64, pl.Int16)
                and col not in ('lat', 'lon')]

print('Kolom numerik yang dianalisis:', numeric_cols)

outlier_summary = []
for col in numeric_cols:
    series = df[col].drop_nulls()
    if len(series) == 0:
        continue
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_outlier = series.filter((series < lower) | (series > upper)).len()
    outlier_summary.append({
        'kolom': col,
        'q1': round(q1, 2),
        'q3': round(q3, 2),
        'iqr': round(iqr, 2),
        'batas_bawah': round(lower, 2),
        'batas_atas': round(upper, 2),
        'n_outlier': n_outlier,
        'pct_outlier': round(n_outlier / len(series) * 100, 2)
    })

outlier_df = pl.DataFrame(outlier_summary)
print(outlier_df)

## 11. Catatan Temuan

*(Isi pengamatan setelah menjalankan sel di atas)*

| Aspek | Temuan |
|-------|--------|
| Jumlah baris | *(isi)* |
| Jumlah kolom | *(isi)* |
| Kolom dengan null terbanyak | *(isi)* |
| Rentang suhu (t) | *(isi, mis. 22–35 °C)* |
| Kelembapan (hu) | *(isi)* |
| Kondisi cuaca paling sering | *(isi)* |
| Outlier signifikan | *(isi)* |
| Temuan lain / anomali | *(isi)* |

---
*Data bersumber dari BMKG — Badan Meteorologi, Klimatologi, dan Geofisika.*